In [ ]:
!pip -q install rdkit==2026.03.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 45.4 MB/s eta 0:00:00


In [ ]:
import re
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import inchi

In [ ]:
# Load the CSV
df = pd.read_csv("Natural Products-Bioactivity Records.csv")
# Preview
print(df.columns)

Index(['ENS_ID', 'Endophyte_Name', 'Endophyte_Type', 'Plant_Material',
       'Fermentation_Condition', 'Fermentation_Condition_Index',
       'Extract_Activity', 'ENC_ID', 'NPs_Name', 'NPs_Content', 'Content_Unit',
       'Potency of Bioactivity', 'Bioactivity', 'ENTO_ID',
       'Target_Organism_Name', 'ENT_ID (ENTP_ID, ENTC_ID, ENTO_ID)',
       'Target_Name', 'Target_Type', 'Assay_Description', 'Activity_Type',
       'NPs_Concentration', 'Concentration_Unit', 'Activity_Values',
       'Control_Name', 'Control_Activity_Values', 'Activity_Unit', 'PMID'],
      dtype='object')


In [ ]:
# Filter to fungal origin only
df = df[df["Endophyte_Type"].str.strip().str.lower().eq("endophytic fungus")]
print(df.shape)
# Count unique fungal
print(df["ENC_ID"].nunique())

(27374, 27)
5855


In [ ]:
# Convert Potency of Bioactivity to binary activity
def potency_to_binary(p):
    if pd.isna(p): return 0
    s = str(p).strip().lower()
    if "inactive" in s: return 0
    # Retain reported bioactivities
    if any(k in s for k in ["active","strong","moderate","weak"]): return 1
    return 0

df["active_flag"] = df["Potency of Bioactivity"].apply(potency_to_binary)

# Count the active fungal

active_fungal = df[(df["active_flag"] == 1)]
print(f"Active compounds count: {active_fungal["ENC_ID"].nunique()}")

Active compounds count: 2302


In [ ]:
# Normalize Bioactivity names
# Count active compounds per bioactivity category
active_df= df[df["active_flag"] == 1]
active_df = active_df.copy()
active_df["Bioactivity_norm"] = (
    active_df["Bioactivity"].astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.lower()
)

active_bioactivities = sorted(active_df["Bioactivity_norm"].unique())

print("Active bioactivities:", len(active_bioactivities))


Active bioactivities: 85


In [ ]:
# Build multilabel table
multi_df = active_df.pivot_table(
    index=["ENC_ID"],
    columns="Bioactivity_norm",
    values="active_flag",
    aggfunc="max",
    fill_value=0
).reset_index()

multi_df_new = multi_df[["ENC_ID"] +  active_bioactivities]

print("Final table shape:", multi_df_new.shape)

Final table shape: (2302, 86)


In [ ]:
# Count bioactivities

bio_cols = active_bioactivities
print("\nCounts per bioactivity:\n")
for col in bio_cols:
    print(f"{col} -> {int(multi_df_new[col].sum())} compounds")


Counts per bioactivity:

11-β hydroxysteroid dehydrogenase-1 inhibitory activity -> 3 compounds
acetylcholinesterase inhibitory activity -> 77 compounds
adenosine deaminase inhibitory activity -> 7 compounds
aldose reductase inhibitory activity -> 8 compounds
allelopathic activity -> 13 compounds
angiotensin converting enzyme inhibitory activity -> 3 compounds
anti-aging activity -> 2 compounds
anti-algal activity -> 4 compounds
anti-allergic activity -> 2 compounds
anti-angiogenic activity -> 4 compounds
anti-atherosclerosis activity -> 1 compounds
anti-bacterial activity -> 774 compounds
anti-bacterial photodynamic activity -> 3 compounds
anti-biofilm activity -> 19 compounds
anti-coagulant activity -> 1 compounds
anti-diabetic activity -> 4 compounds
anti-feedant activity -> 13 compounds
anti-fouling activity -> 4 compounds
anti-fungal activity -> 538 compounds
anti-hyperlipidemic activity -> 9 compounds
anti-inflammatory activity -> 240 compounds
anti-leishmanial activity -> 28 co

In [ ]:
# Create ENC_ID -> Endophyte_Name mapping
source_map = (
    active_df.groupby("ENC_ID")["Endophyte_Name"]
    .apply(lambda x: " || ".join(sorted(x.dropna().astype(str).unique())))
    .reset_index()
)

# Create ENC_ID -> PMID mapping
pmid_map = (
    active_df.groupby("ENC_ID")["PMID"]
    .apply(lambda x: " || ".join(sorted(x.dropna().astype(str).unique())))
    .reset_index()
)

# Merge metadata
multi_df_new = multi_df_new.merge(source_map, on="ENC_ID", how="left")
multi_df_new = multi_df_new.merge(pmid_map, on="ENC_ID", how="left")

# Reorder columns
multi_df_new = multi_df_new[
    ["ENC_ID", "Endophyte_Name", "PMID"] + active_bioactivities
]

In [ ]:
multi_df_new.shape

(2302, 88)

In [ ]:
# Load NP General Information file
meta = pd.read_csv("Natural Products General Information.csv")
print(meta.columns)

Index(['ENC_ID', 'NPs_Name', 'Synonyms', 'Molecular_Formula',
       'Molecular_Weight', 'Smiles', 'Inchi', 'Inchikey', 'IUPAC', 'Alog P',
       'TPSA', 'H_Bond_Donor', 'H_Bond_Acceptor', 'Rotatable_Bond',
       'Heavy_Atom', 'Aromatic_Rings', 'QED_Weighted', 'Lipskin',
       'PubChem_CID', 'CAS', 'ChEMBL_ID'],
      dtype='object')


In [ ]:
# Keep only the needed columns from metadata
meta_row = meta[["ENC_ID", "NPs_Name", "Smiles", "PubChem_CID"]]

In [ ]:
# Merge
merged = multi_df_new.merge(meta_row, on="ENC_ID", how="left")

total = len(merged)
print(f"Total compounds in labels: {total}")

Total compounds in labels: 2302


In [ ]:
# Put ID columns first, then labels
id_cols = ["ENC_ID", "NPs_Name", "Smiles", "PubChem_CID"]
label_cols = [c for c in merged.columns if c not in id_cols]
merged = merged[id_cols + label_cols]
merged

,ENC_ID,NPs_Name,Smiles,PubChem_CID,Endophyte_Name,PMID,11-β hydroxysteroid dehydrogenase-1 inhibitory activity,acetylcholinesterase inhibitory activity,adenosine deaminase inhibitory activity,aldose reductase inhibitory activity,...,topoisomerase inhibitory activity,tremorgenic activity,trypanocidal activity,tumor necrosis factor-α inhibitory activity,tyrosine kinase inhibitory activity,urease inhibitory activity,xanthine oxidase inhibitory activity,α-amylase inhibitory activity,α-glucosidase inhibitory activity,β-glucuronidase inhibitory activity
0,ENC000002,"3,4-Dihydroxybenzoic acid",C1=CC(=C(C=C1C(=O)O)O)O,72.0,Phyllosticta capitalensis HL10,31441681,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,ENC000005,4-Hydroxybenzaldehyde,C1=CC(=CC=C1C=O)O,126.0,Fusarium solani CARE-1,30294556,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ENC000006,4-Hydroxyphenylacetic acid,C1=CC(=CC=C1CC(=O)O)O,127.0,Colletotrichum gloeosporioides,25421415,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,ENC000007,4-Hydroxybenzoic acid,C1=CC(=CC=C1C(=O)O)O,135.0,Trichoderma erinaceum,35583291,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,ENC000011,Adenine,C1=NC2=NC=NC(=C2N1)N,190.0,Aspergillus amstelodami AUMC (EGY),31241824,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2297,ENC006366,NaN,NaN,NaN,Colletotrichum dematium CR-17,18599825,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2298,ENC006419,NaN,NaN,NaN,Alternaria longipes VITN14G,30321457,0,0,0,0,...,0,0,0,0,0,0,0,1,1,0
2299,ENC006503,NaN,NaN,NaN,Cladosporium sphaerospermum WBS017,32101782,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2300,ENC006536,NaN,NaN,NaN,Botryosphaeria dothidea KJ-1,24689437,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Remove rows where SMILES is missing in meta file
before = len(merged)
merged = merged[merged["Smiles"].notna()].reset_index(drop=True)
after = len(merged)

print(f"Removed {before - after} compounds with no SMILES.")
print(f"Final dataset size: {after} compounds")

Removed 58 compounds with no SMILES.
Final dataset size: 2244 compounds


In [ ]:
# Function: SMILES -> canonical SMILES
def canonicalize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
    except:
        return None  # invalid SMILES

In [ ]:
# Function: SMILES -> InChIKey
def smiles_to_inchikey(smi):
    try:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            return None
        return inchi.MolToInchiKey(mol)
    except:
        return None

In [ ]:
# Compute canonical smiles
merged["canonical_smiles"] = merged["Smiles"].apply(canonicalize_smiles)
# Report how many succeeded
success_CS = merged["canonical_smiles"].notna().sum()
total = len(merged)
print(f"Generated canonical smiles for {success_CS}/{total} compounds ({success_CS/total*100:.1f}%).")

Generated canonical smiles for 2244/2244 compounds (100.0%).


In [ ]:
# Compute InChIKeys
merged["InChIKey"] = merged["Smiles"].apply(smiles_to_inchikey)
# Report how many succeeded
success_InChI = merged["InChIKey"].notna().sum()
total = len(merged)
print(f"Generated InChIKeys for {success_InChI}/{total} compounds ({success_InChI/total*100:.1f}%).")

Generated InChIKeys for 2244/2244 compounds (100.0%).


In [ ]:
merged.to_csv("EMNPD_processed.csv", index=False)